# Controlling MODFLOW 6 with the API — E. Streamflow augmentation at the coast

This is a variant of part E of the MODFLOW 6 application programming interface
(**API**) series — see [**A. Basic usage**](mf6-api-a-basic.ipynb) for the API
lifecycle and
[**C. Adjusting recharge with a callback**](mf6-api-c-recharge-callback.ipynb)
for the callback mechanism this example builds on.

[**E. Streamflow augmentation**](mf6-api-e-streamflow-augmentation.ipynb) runs an
operating rule on the inland valley: pump a well whenever flow at the stream
gauge falls below a threshold, and add the pumped water to the stream. The rule
works, but not well. The well is close enough to the stream that pumping it takes
back most of what it adds.

Run the same rule on the coastal valley, where the aquifer also meets the sea,
and the answer changes. The well now has a second source to draw on, so less of
what it pumps comes out of the stream and more of the augmented water stays
there.

## What this notebook covers

Run an operating rule on a valley with a coast, and measure how much of the
pumped water the stream keeps.

By the end of this notebook you will be able to:

- give a constant-density model a saltwater coast by raising its boundary heads
  to **equivalent freshwater heads**,
- see what that coast does to streamflow before any well is switched on,
- read streamflow from a running model and set a pumping rate from it, taking
  care to read the reach the gauge reports,
- compare an operating rule updated once per stress period with one updated every
  outer iteration, and
- measure the flow the stream keeps per unit pumped, and check it against the
  capture fraction the adjoint analysis gives for the same well.

## Imports and setup

Import FloPy, the `modflowapi` interface, and the plotting and data libraries.
`%matplotlib inline` has to come first, or only the first figure drawn inside a
`flopy.plot.styles` context is rendered.

In [ ]:
%matplotlib inline

import pathlib as pl
import shutil

import flopy
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mf6_notebook_helpers import (
    coastal_ghb_data,
    find_mf6_libraries,
    seconds_per_day,
)
from modflowapi import Callbacks, run_simulation

### Locate the MODFLOW 6 library

The API drives MODFLOW 6 through its compiled shared library (`libmf6`), not the
`mf6` command-line executable. `find_mf6_libraries()` locates the library and the
`mf6` executable inside the active pixi environment; both are needed, the
executable for the ordinary baseline runs and the library for the API runs.

In [ ]:
lib_name, mf6_exe = find_mf6_libraries()
print(f"library:    {lib_name.name}")
print(f"executable: {mf6_exe.name}")

## Build the two valleys

Both models are the **advanced** synthetic valley, which represents the valley
with streamflow routing (**SFR**) for the river, a lake (**LAK**),
unsaturated-zone flow (**UZF**), and the water mover (**MVR**). SFR is what makes
this notebook possible: the rule needs a routed streamflow at a gauge to control
on. The 21 stress periods are a long steady spin-up followed by 20 annual
periods, in feet and days.

The coastal valley adds a general-head boundary (**GHB**) along its southern row,
built in [mf6-coastal-ghb](mf6-coastal-ghb.ipynb). Seawater is denser than fresh
groundwater, so a column of it presses down harder than the same column of fresh
water would. These models carry a single constant fluid density, and the way to
represent the weight of the sea in such a model is to raise the boundary head
until a column of fresh water would press down just as hard. That raised head is
the **equivalent freshwater head**, and it grows with depth below sea level.
`coastal_ghb_data(gwf, equivalent_freshwater=True)` rebuilds the boundary with
those heads.

`build()` below loads a shipped model, switches the prediction well off so the
run is a clean baseline, optionally rebuilds the coastal boundary, and writes and
runs it. Both valleys are built the same way so the only difference between them
is the coast.

In [ ]:
DATA = pl.Path("../data/synthetic-valley")
MODELS = pl.Path("models")
start_date = pd.to_datetime("1962-01-01 00:00:00")


def build(source, workspace, equivalent_freshwater=False, run=True):
    """Write a synthetic-valley model with the prediction well switched off."""
    workspace = MODELS / workspace
    if workspace.exists():
        shutil.rmtree(workspace)
    sim = flopy.mf6.MFSimulation.load(
        sim_name="sv",
        sim_ws=str(DATA / source),
        exe_name=mf6_exe,
        write_headers=False,
        verbosity_level=0,
    )
    gwf = sim.get_model("sv")

    # the shipped prediction well carries a rate in every period it appears in;
    # zero them all, so the callback is the only thing that ever switches it on
    prediction = gwf.get_package("prediction")
    rates = {}
    for kper in range(sim.tdis.nper.array):
        period = prediction.stress_period_data.get_data(kper)
        if period is None:
            continue
        period["q"] = 0.0
        rates[kper] = period
    prediction.stress_period_data.set_data(rates)

    if equivalent_freshwater:
        spd, _ = coastal_ghb_data(gwf, equivalent_freshwater=True)
        gwf.get_package("ghb-1").stress_period_data.set_data({0: spd})

    sim.set_sim_path(str(workspace))
    sim.write_simulation(silent=True)
    if run:
        success, buff = sim.run_simulation(silent=True)
        if not success:
            raise RuntimeError("\n".join(buff[-20:]))
    return sim, workspace


coastal_sim, coastal_ws = build(
    "synthetic-valley-coastal-advanced-annual",
    "api-e-coastal-base",
    equivalent_freshwater=True,
)
inland_sim, inland_ws = build(
    "synthetic-valley-advanced-annual", "api-e-coastal-inland-base"
)

ghb = coastal_sim.get_model("sv").get_package("ghb-1").stress_period_data.get_data(0)
print("coastal boundary head by layer, ft above sea level")
for k in sorted({cellid[0] for cellid in ghb["cellid"]}):
    layer = ghb[[cellid[0] == k for cellid in ghb["cellid"]]]
    print(
        f"  layer {k + 1}: {layer['bhead'].min():6.3f} to {layer['bhead'].max():6.3f}"
        f"   at elevations {layer['elevation'].min():8.1f} to"
        f" {layer['elevation'].max():7.1f} ft"
    )

**What to look for.** The boundary head rises from 0.061 ft where the coastal
cells sit 2.5 ft below sea level to between 3.1 and 5.8 ft in layer 5, where they
sit 126 to 236 ft down. Every one of those heads is above sea level, so the sea
resists outflow from the aquifer and, where the aquifer head is low enough,
drives water inland.

### Look at the coastal valley

Map the features the rule works with: the stream, the coast along the southern
row, the two production wells, and the prediction well the rule switches on. Cell
IDs are `(layer, row, column)` counting from zero, as everywhere in FloPy.

In [ ]:
gwf = coastal_sim.get_model("sv")
nlay, nrow, ncol = gwf.dis.nlay.data, gwf.dis.nrow.data, gwf.dis.ncol.data
xc, yc = gwf.modelgrid.xcellcenters, gwf.modelgrid.ycellcenters
coast_rc = sorted({(cellid[1], cellid[2]) for cellid in ghb["cellid"]})

with flopy.plot.styles.USGSMap():
    fig, ax = plt.subplots(figsize=(4.5, 7), layout="constrained")
    mv = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
    heads = gwf.output.head().get_data()
    heads[heads == 1e30] = np.nan
    cb = mv.plot_array(heads[0])
    contours = mv.contour_array(heads[0], levels=np.arange(0, 15, 2), colors="black")
    ax.clabel(contours, contours.levels, fmt="%.0f", fontsize=8)
    mv.plot_bc("SFR", color="tab:cyan")
    mv.plot_bc("LAK", color="tab:blue")
    ax.plot(
        [xc[i, j] for i, j in coast_rc],
        [yc[i, j] for i, j in coast_rc],
        "s",
        color="tab:green",
        ms=6,
        label="coast",
    )
    mv.plot_bc("WELL", package=gwf.pwell, plotAll=True, color="black", kper=12)
    mv.plot_bc("WELL", package=gwf.prediction, plotAll=True, color="red", kper=12)
    ax.plot([], [], "s", color="tab:cyan", label="stream")
    ax.plot([], [], "s", color="tab:blue", label="lake")
    ax.plot([], [], "s", color="black", label="production wells")
    ax.plot([], [], "s", color="red", label="prediction well")
    ax.legend(loc="upper right", fontsize=8)
    ax.set_xlabel("x, in feet")
    ax.set_ylabel("y, in feet")
    fig.colorbar(cb, ax=ax, shrink=0.35, label="head, in feet")
    ax.set_title("Coastal synthetic valley, layer 1")

**What to look for.** Heads in the top layer fall from the lake in the north
toward the stream and the coast in the south. The prediction well (red) sits in
the southern third of the valley, 2,500 ft from the coast and 3,500 ft from the
nearest stream reach, so both the stream and the sea are within reach of it. That
is the whole of the difference from the inland valley, where only the stream is.

## What the coast does before any well is switched on

Read the gauge in both baselines and compare them. The gauge is the SFR
observation `riv-flow`, defined in `sv.sfr.obs` as the downstream flow of reach
17 of 18. The model's time unit is days, so MODFLOW reports flows in cubic feet
per day; divide by `-86400` to get cubic feet per second (**cfs**) with a
positive-discharge convention.

In [ ]:
min_flow = 15.0  # target minimum streamflow at the gauge (cfs)
aug_start = 10  # the rule runs in the stress periods after this one


def gauge_flow(sim):
    """Gauge flow per stress period in cfs, indexed by the model's own dates."""
    observations = (
        sim.get_model("sv").sfr.output.obs().get_dataframe(start_datetime=start_date)
    )
    return observations["RIV-FLOW"] / -seconds_per_day


coastal_base = gauge_flow(coastal_sim)
inland_base = gauge_flow(inland_sim)
years = coastal_base.index  # the end of each stress period
window = slice(aug_start + 1, None)  # the periods the rule runs in

print("             start    min    mean   below target")
for label, series in (("coastal", coastal_base), ("inland", inland_base)):
    during = series.iloc[window]
    print(
        f"  {label:9s} {series.iloc[0]:6.2f} {during.min():6.2f}"
        f" {during.mean():7.2f}   {int((during < min_flow).sum())} of {len(during)}"
    )
print(f"\n  the coast adds {(coastal_base - inland_base).mean():.2f} cfs on average")

In [ ]:
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
    # each value is the flow over a whole stress period, so hold it across the
    # period rather than sloping between the sampled points
    step = "steps-pre"
    ax.plot(
        years,
        coastal_base,
        color="tab:green",
        lw=1.8,
        drawstyle=step,
        label="coastal valley",
    )
    ax.plot(
        years,
        inland_base,
        color="tab:brown",
        lw=1.8,
        drawstyle=step,
        label="inland valley",
    )
    ax.axhline(min_flow, color="black", ls=":", lw=1.0, label="target")
    ax.axvspan(
        years[aug_start],
        years[-1],
        color="cyan",
        alpha=0.2,
        lw=0,
        label="augmentation period",
    )
    ax.set_xlabel("year")
    ax.set_ylabel("gauge discharge, in cfs")
    ax.set_ylim(8, 20)
    ax.legend(loc="upper right", ncols=2, fontsize=8)
    ax.set_title("Baseline gauge flow, prediction well off")

**What to look for.** The coastal gauge runs above the inland gauge in every
year, by 1.66 cfs on average. An equivalent freshwater head puts the sea above
the aquifer head over much of the coast, so the coast is a net source to the
valley rather than a drain on it, and the extra water leaves through the stream.

The coast makes the target easier but does not meet it. The coastal baseline is
below 15 cfs in 8 of the 10 augmentation years compared to the inland valley's 9,
and its lowest year is 12.49 cfs compared to 10.41 cfs. The same 15 cfs target
used in
[mf6-api-e](mf6-api-e-streamflow-augmentation.ipynb) is kept here, so the two
notebooks can be read against each other.

## The operating rule

When flow at the gauge falls below `min_flow`, switch the prediction well on at a
rate proportional to the shortfall, capped at a maximum, and add that same rate
as stream inflow at the upstream end of the river. When the gauge is at or above
the target, switch the well off.

The rule reads the reach the gauge reports, `gauge_reach`, out of the SFR
downstream-flow array. The reach below the gauge carries about 1.5 cfs more, so a
rule reading the last reach instead would believe the target was met while the
gauge said otherwise.

`update_each_iteration` chooses between two ways of applying the rule:

- **False** sets the rate once per stress period, at `Callbacks.stress_period_start`,
  from the flow at the end of the previous period.
- **True** also recomputes it at the start of every outer iteration
  (`Callbacks.iteration_start`), from the flow in the current solution, so the
  rate converges along with the heads instead of lagging a period behind.

The per-iteration rate is **under-relaxed**: each outer iteration it moves only a
fraction `relax` of the way toward the rate the current flow calls for. Pumping
the well draws water from the stream, which lowers the flow the rule is reading,
so an un-relaxed rate chases its own effect and does not settle.

In [ ]:
gauge_reach = 16  # zero-based; the gauge is reach 17 of 18 in sv.sfr.obs

# required rate = GAIN * (shortfall below min_flow), capped at RATE_CAP
GAIN = 2.5  # dimensionless
RATE_CAP = 2.0e6  # ft^3/d
relax = 0.5  # under-relaxation for the per-iteration update (0 < relax <= 1)

aug_rate = 0.0  # ft^3/d, kept across outer iterations so it can be relaxed
update_each_iteration = False


def callback_function(sim, callback_step):
    """Pump the prediction well to hold the gauge at min_flow."""
    global aug_rate
    ml = sim.sv

    def required_rate():
        """Rate (ft^3/d) the flow now at the gauge calls for."""
        flow = sim.mf6.get_value(sim.mf6.get_var_address("DSFLOW", "SV", "SFR-1"))
        q = float(flow[gauge_reach]) / seconds_per_day
        if q < min_flow:
            return min(GAIN * (min_flow - q) * seconds_per_day, RATE_CAP)
        return 0.0

    def apply_rate(rate):
        """Pump at `rate` and put the same rate into the top of the stream."""
        ml.prediction.stress_period_data["q"] = -rate
        inflow = sim.mf6.get_value_ptr(sim.mf6.get_var_address("INFLOW", "SV", "SFR-1"))
        inflow[0] = rate

    if callback_step == Callbacks.stress_period_start and sim.kper > aug_start:
        aug_rate = required_rate()
        apply_rate(aug_rate)

    if callback_step == Callbacks.iteration_start:
        if update_each_iteration and sim.kper > aug_start:
            aug_rate += relax * (required_rate() - aug_rate)
            aug_rate = min(max(aug_rate, 0.0), RATE_CAP)
            apply_rate(aug_rate)

### Run the rule both ways

Run the coastal model twice, once with each update strategy, each in its own
workspace so the baseline output is not overwritten. `run_simulation()` drives
MODFLOW 6 through the library, calling `callback_function` at every stage.

`pumping_rate()` reads what the well actually pumped back out of the cell-by-cell
budget rather than trusting the rate the callback asked for, which is the check
that the rule did what it meant to.

In [ ]:
def pumping_rate(sim):
    """Prediction-well rate per stress period in cfs, from the budget file."""
    budget = sim.get_model("sv").output.budget()
    rates = []
    for totim in budget.get_times():
        record = budget.get_data(totim=totim, paknam2="prediction")[0]
        rates.append(0.0 if len(record) == 0 else float(record["q"][0]))
    return np.array(rates) / -seconds_per_day


def run_rule(source, workspace, equivalent_freshwater, per_iteration):
    """Run one model under the operating rule and return (gauge flow, rate)."""
    global aug_rate, update_each_iteration
    sim, ws = build(source, workspace, equivalent_freshwater, run=False)
    aug_rate = 0.0
    update_each_iteration = per_iteration
    run_simulation(lib_name, ws, callback_function, verbose=False)
    return gauge_flow(sim), pumping_rate(sim)


coastal = {}
for label, per_iteration in (("per stress period", False), ("per iteration", True)):
    coastal[label] = run_rule(
        "synthetic-valley-coastal-advanced-annual",
        f"api-e-coastal-{per_iteration}",
        True,
        per_iteration,
    )
print("done")

In [ ]:
with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["a"], ["b"]], figsize=(9, 6), layout="constrained", sharex=True
    )
    colors = {"per stress period": "tab:orange", "per iteration": "tab:blue"}

    ax = axd["a"]
    step = "steps-pre"  # a gauge value is the flow over a whole stress period
    ax.plot(
        years, coastal_base, color="tab:green", lw=1.8, drawstyle=step, label="baseline"
    )
    for label, (flow, _) in coastal.items():
        ax.plot(years, flow, color=colors[label], lw=1.8, drawstyle=step, label=label)
    ax.axhline(min_flow, color="black", ls=":", lw=1.0, label="target")
    ax.axvspan(years[aug_start], years[-1], color="cyan", alpha=0.2, lw=0)
    ax.set_ylabel("gauge discharge, in cfs")
    ax.set_ylim(10, 20)
    ax.legend(loc="upper left", ncols=4, fontsize=8)

    # the rate is held constant over a stress period, so draw it as a bar
    # spanning that period rather than as a line between sampled points
    ax = axd["b"]
    width = 150.0  # days, the unit of a matplotlib date axis
    offsets = {"per stress period": -0.5 * width, "per iteration": 0.5 * width}
    for label, (_, rate) in coastal.items():
        ax.bar(
            mdates.date2num(years) + offsets[label],
            rate,
            width,
            color=colors[label],
            edgecolor="black",
            lw=0.4,
            label=label,
        )
    ax.axvspan(years[aug_start], years[-1], color="cyan", alpha=0.2, lw=0, zorder=0)
    ax.set_ylabel("augmentation rate, in cfs")
    ax.set_xlabel("year")
    ax.set_ylim(0, 5)
    for panel, ax in axd.items():
        flopy.plot.styles.heading(ax=ax, letter=panel)

**What to look for.** Panel a is the gauge and panel b the rate the rule chose.
Both strategies lift the gauge, and they fail in different ways.

Updating once per stress period (orange) sets the rate from the previous year's
flow, so the rule is always answering last year's question. It overshoots to 18.8
cfs in one year and switches off entirely in the next, and the rate alternates
between 0 and 4.47 cfs. Updating every outer iteration (blue) reads the flow in
the solution it is part of, and its rate never exceeds 2.44 cfs, dropping to zero
only in the two years the gauge is already above the target. Its gauge flow never
falls below 14.02 cfs, compared with 12.43 cfs for the per-period rule and 12.49
cfs with no rule at all.

The steadier rule holds a lower mean, 14.78 cfs compared to 15.06 cfs. Updating
once per period pumps harder and wastes some of it; updating every iteration
pumps only what the shortfall calls for and holds the flow closer to the target
year on year.

## How much of the pumped water does the stream keep?

Pumping the prediction well lowers heads near the stream, so part of the water
added at the top of the river leaks straight back into the aquifer. What the
gauge gains per unit pumped is what the rule is actually worth.

Measure it as the total lift at the gauge over the augmentation period divided by
the total pumped, and run the inland valley under the same rule to compare. The
difference between the two is the coast.

In [ ]:
inland = {}
for label, per_iteration in (("per stress period", False), ("per iteration", True)):
    inland[label] = run_rule(
        "synthetic-valley-advanced-annual",
        f"api-e-coastal-inland-{per_iteration}",
        False,
        per_iteration,
    )

rows = []
for valley, baseline, results in (
    ("coastal", coastal_base, coastal),
    ("inland", inland_base, inland),
):
    for label, (flow, rate) in results.items():
        during = flow.iloc[window]
        lift = (flow - baseline).iloc[window]
        pumped = rate[window]
        rows.append(
            {
                "valley": valley,
                "update": label,
                "min gauge (cfs)": during.min(),
                "mean gauge (cfs)": during.mean(),
                "years below target": int((during < min_flow).sum()),
                "total pumped (cfs-yr)": pumped.sum(),
                "total lift (cfs-yr)": lift.sum(),
                "kept per unit pumped": lift.sum() / pumped.sum(),
            }
        )
summary = pd.DataFrame(rows).set_index(["valley", "update"])
summary.round(3)

**What to look for.** The coastal stream keeps 0.645 of every unit pumped under
the per-period rule and 0.634 under the per-iteration rule. The inland stream
keeps 0.281 and 0.294. The same well, the same rule, and the same target return
more than twice as much water to the stream once the valley has a coast.

The cost side says the same thing. The inland rule pumps 44.2 cfs-years to lift
the gauge by 12.4 cfs-years; the coastal per-iteration rule pumps 11.6 to lift it
by 7.4. Roughly a quarter of the water, for more than half the lift.

[mf6-adj-coastal-capture](mf6-adj-coastal-capture.ipynb) explains the number
without running the rule at all. Adjoint-state sensitivity analysis on this same
coastal model gives the prediction well a stream capture fraction of 0.345: for
every unit it pumps, 0.345 comes out of the stream and most of the rest comes
from the sea. A rule that adds one unit at the top of the river and pumps one
unit out of the aquifer should leave 1 − 0.345 = 0.655 at the gauge, and the
measurement here is 0.645. The adjoint predicted what this operating rule is
worth from a single backward solve, with no operating rule in the model at all.

## Recap

- A saltwater coast in a constant-density model is a general-head boundary at the
  **equivalent freshwater head**, which rises with depth below sea level.
- That coast is a net source to this valley, and it lifts the gauge by 1.66 cfs
  on average before any well is switched on.
- The same operating rule as
  [mf6-api-e](mf6-api-e-streamflow-augmentation.ipynb) runs here through a
  `modflowapi` callback, reading the SFR reach the gauge reports and writing both
  the well rate and the stream inflow.
- Updating the rate every outer iteration, under-relaxed, holds the gauge steadier
  than updating once per stress period, and pumps less to do it.
- The coastal stream keeps 0.645 of every unit pumped, compared with 0.281
  inland, because the well draws most of its water from the sea rather than from
  the stream.
- That 0.645 matches the 1 − 0.345 the adjoint capture fraction predicts, so a
  sensitivity analysis can price an operating rule before the rule is written.